# Random forest explanations
Explain the saved optimized model for `adverse_365` (class 1). Set `BASELINE` to A or B and run all cells after Person 2 supplies the matching model file. Install the project requirements in the notebook environment first.

The notebook compares feature rankings, shows directional SHAP effects, and explores three clinical features. Results use held-out records and a fixed random seed.

In [ ]:
from pathlib import Path
import hashlib
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

BASELINE = "B"
SEED = 42
PERMUTATION_ROWS = 5000
SHAP_ROWS = 500
TOP_N = 10
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "models").is_dir())
stem = f"random_forest_baseline_{BASELINE.lower()}"
model_path = ROOT / "models" / f"{stem}_best.pkl"
if not model_path.is_file():
    raise FileNotFoundError(f"Ask Person 2 for the final trained model: {model_path}")
metadata = json.loads((ROOT / "models" / f"{stem}_metadata.json").read_text())
# Load only the trusted model supplied by the project team.
with model_path.open("rb") as handle:
    model = pickle.load(handle)
if not isinstance(model, RandomForestClassifier):
    raise TypeError("Expected the optimized RandomForestClassifier saved by the training notebook.")
if metadata["positive_class"] != 1 or list(model.classes_) != [0, 1]:
    raise ValueError("Expected outcome classes 0 and 1, with adverse outcome coded as 1.")

## Recreate the held-out split
Check the source hashes and feature order against training metadata so the explanations use the same data and split.

In [ ]:
data_dir = ROOT / "processed_output"
for filename, expected in metadata["source_sha256"].items():
    with (data_dir / filename).open("rb") as handle:
        actual = hashlib.file_digest(handle, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"Training data mismatch for {filename}.")
X = pd.read_csv(data_dir / "X_processed.csv", index_col=0)[metadata["feature_columns"]]
y_frame = pd.read_csv(data_dir / "y_processed.csv", index_col=0)
if y_frame.shape[1] != 1 or not X.index.equals(y_frame.index):
    raise ValueError("Feature and outcome rows must align.")
y = y_frame.iloc[:, 0]
if list(model.feature_names_in_) != list(X.columns):
    raise ValueError("Model and metadata feature orders differ.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=metadata["test_size"], random_state=metadata["random_state"],
    stratify=y if metadata["stratified_split"] else None,
)
X_eval, y_eval = X_test, y_test
if len(X_test) > PERMUTATION_ROWS:
    X_eval, _, y_eval, _ = train_test_split(
        X_test, y_test, train_size=PERMUTATION_ROWS, stratify=y_test, random_state=SEED,
    )
print(f"Baseline {BASELINE}, target {metadata['target']}, evaluation rows {len(X_eval):,}")
print(f"Sample average precision {average_precision_score(y_eval, model.predict_proba(X_eval)[:, 1]):.4f}")

## Compare feature importance
Gini importance measures how much splits improve purity. It can favor features with many distinct values. Permutation importance measures the drop in held-out average precision after shuffling a feature. Larger drops suggest greater reliance on that feature. Error bars show variation across five shuffles, and are not confidence intervals. Correlated features can share importance.

In [ ]:
permutation = permutation_importance(
    model, X_eval, y_eval, scoring="average_precision", n_repeats=5,
    random_state=SEED, n_jobs=1,
)
importance = pd.DataFrame({
    "Gini importance": model.feature_importances_,
    "Permutation AP drop": permutation.importances_mean,
    "Permutation SD": permutation.importances_std,
}, index=X.columns).sort_values("Permutation AP drop", ascending=False)
display(importance.head(TOP_N).round(5))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
importance.nlargest(TOP_N, "Gini importance")["Gini importance"].sort_values().plot.barh(ax=axes[0])
top = importance.head(TOP_N).sort_values("Permutation AP drop")
axes[1].barh(top.index, top["Permutation AP drop"], xerr=top["Permutation SD"])
axes[0].set_title("Built-in Gini importance")
axes[1].set_title("Held-out permutation importance")
axes[1].set_xlabel("Average precision drop")
axes[1].axvline(0, color="gray", linewidth=1)
plt.tight_layout()
plt.show()

## Directional SHAP summary
Positive SHAP values increase the predicted probability of an adverse outcome. Negative values decrease it. Red points represent higher feature values and blue points represent lower values. Each point is one held-out record.

Use 100 training records as the background and up to 500 held-out records for explanations. [TreeExplainer documentation](https://shap.readthedocs.io/en/stable/generated/shap.TreeExplainer.html).

In [ ]:
X_shap = X_eval.sample(n=min(SHAP_ROWS, len(X_eval)), random_state=SEED)
background = X_train.sample(n=min(100, len(X_train)), random_state=SEED)
explainer = shap.TreeExplainer(
    model, data=background, feature_perturbation="interventional", model_output="probability",
)
explanation = explainer(X_shap)[:, :, 1]
np.testing.assert_allclose(
    explanation.base_values + explanation.values.sum(axis=1),
    model.predict_proba(X_shap)[:, 1], atol=1e-5,
)
shap.plots.beeswarm(explanation, max_display=TOP_N)
shap_rank = pd.Series(np.abs(explanation.values).mean(axis=0), index=X.columns).sort_values(ascending=False)

## Clinical feature effects
Show the three available clinical features with the largest mean absolute SHAP values. The horizontal axis uses processed values. Age, BMI, and dialysis duration may already be standardized, so these axes cannot establish thresholds in years or BMI units. Dialysis duration measures time on dialysis at listing. Lab values and waitlist duration are absent from the saved feature lists.

In [ ]:
clinical_features = [
    "INIT_AGE", "BMI_TCR_CLEAN", "dialysis_years_at_listing",
    "INIT_CPRA_CLEAN_MAPPED", "FUNC_STATUS_PERCENT_MAPPED", "dialysis_at_listing",
]
top_clinical = shap_rank[shap_rank.index.isin(clinical_features)].head(3).index
for feature in top_clinical:
    shap.plots.scatter(explanation[:, feature], show=False)
    plt.xlabel(f"{feature} (processed value)")
    plt.ylabel("SHAP contribution to adverse probability")
    plt.tight_layout()
    plt.show()

## Stakeholder takeaways
The following summary is calculated when the notebook runs. Read the dependence plots for thresholds or changes in direction. Differences between high and low values summarize the sample and can hide nonlinear patterns.

In [ ]:
display(Markdown(
    f"The largest held-out performance drops are associated with **{', '.join(importance.head(3).index)}**. "
    f"The largest average SHAP contributions come from **{', '.join(shap_rank.head(3).index)}**. "
    "Check the signs and error bars before treating a permutation ranking as meaningful."
))
rows = []
for feature in top_clinical:
    values = X_shap[feature]
    low, high = values.quantile([0.25, 0.75])
    effects = explanation[:, feature].values
    rows.append({
        "Clinical feature": feature,
        "Low-value mean SHAP (percentage points)": 100 * effects[values <= low].mean(),
        "High-value mean SHAP (percentage points)": 100 * effects[values >= high].mean(),
    })
display(pd.DataFrame(rows).round(3))

These results describe model associations. Causal conclusions require separate evidence. Correlated inputs and one-hot categories can make individual feature rankings unstable. The sample sizes limit how much rare patterns appear in the plots.

The saved evaluation reports zero adverse-case recall at the default 0.5 cutoff for both baselines. Keep that performance limitation alongside these explanations. Upstream preprocessing and repeated-patient handling are also unresolved in the training workflow.